#### ***Deep Learning Fundamentals Day 78 📊***
***
-  ***RNN Project***

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
df.shape

(50000, 2)

In [3]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df.shape

(49582, 2)

- ***Preprocessing***

In [6]:
# converting to lowercase
df["review"] = df['review'].str.lower()

In [7]:
# removing urls
import re

def remove_url(text):
     text = re.sub(r"http\S+","",text)  # (pattern,repl,string)
     return text

df["review"] = df['review'].apply(remove_url)

In [9]:
# remove punctuations

import re
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [10]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [11]:
# removing html tags

def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [12]:
# Removing Stop words
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gzbra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gzbra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gzbra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
     tokens = word_tokenize(text)
     stop_words = stopwords.words('english')

     for word in tokens:
          if word in stop_words:
               text = text.replace(word,"")
     return text
df["review"] = df["review"].apply(remove_stopwords)

In [15]:
# running -> run
# played -> play
# PorterStemming

from nltk.stem import PorterStemmer
def stemming(text):
     ps = PorterStemmer()
     stemmed_words = []

     tokens = word_tokenize(text)
     for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
     return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)


In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [16]:
# Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [18]:
y = df["sentiment"]

In [19]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

In [21]:
# Dataset & Data Loaders
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
X_train.shape

(39665, 5000)

In [23]:
X_test.shape

(9917, 5000)

In [24]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [25]:
# Build RNN

import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [26]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [27]:
# Train RNN
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.30194592475891113
epoch = 2/10 and loss = 0.37575051188468933
epoch = 3/10 and loss = 0.1960265338420868
epoch = 4/10 and loss = 0.22968973219394684
epoch = 5/10 and loss = 0.2224193513393402
epoch = 6/10 and loss = 0.3423786461353302
epoch = 7/10 and loss = 0.2832048237323761
epoch = 8/10 and loss = 0.1512407511472702
epoch = 9/10 and loss = 0.21808667480945587
epoch = 10/10 and loss = 0.2759895622730255


In [28]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.61056771200968
